# Leyendo la base de datos

In [ ]:
import pandas as pd
dataset_entrenamiento = pd.read_csv('../data/train_clean.csv')
dataset_prueba = pd.read_csv('../data/test_clean.csv')
dataset_oot = pd.read_csv('../data/oot_clean.csv')

# Creando X & Y

In [ ]:
X_entrenamiento = dataset_entrenamiento.drop(columns=['es_fraude'])
y_entrenamiento = dataset_entrenamiento['es_fraude']

X_prueba = dataset_prueba.drop(columns=['es_fraude'])
y_prueba = dataset_prueba['es_fraude']

X_oot = dataset_oot.drop(columns=['es_fraude'])
y_oot = dataset_oot['es_fraude']

print("Dimensiones:")
print(f"Entrenamiento -> X: {X_entrenamiento.shape}, y: {y_entrenamiento.shape}")
print(f"Prueba        -> X: {X_prueba.shape}, y: {y_prueba.shape}")
print(f"OOT           -> X: {X_oot.shape}, y: {y_oot.shape}")

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Función objetivo

In [ ]:
def objective_rf(trial):
    
    n_estimators = trial.suggest_int('n_estimators', 50, 150, step=50)
    max_depth = trial.suggest_int('max_depth', 5, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 10, 50)
    class_weight = trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample'])
    
    modelo_rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        class_weight=class_weight,
        n_jobs=-1, 
        random_state=42
    )
    
    modelo_rf.fit(X_entrenamiento, y_entrenamiento)
    
    y_pred_proba = modelo_rf.predict_proba(X_prueba)[:, 1]
    
    auc = roc_auc_score(y_prueba, y_pred_proba)
    return auc

print("Iniciando búsqueda con Optuna")

estudio_rf = optuna.create_study(direction='maximize')

estudio_rf.optimize(objective_rf, n_trials=10)

print("\n" + "="*50)
print("RESULTADOS")
print("="*50)
print(f"Mejor ROC-AUC alcanzado: {estudio_rf.best_value:.5f}")
print("\nLos hiperparámetros ganadores fueron:")
for clave, valor in estudio_rf.best_params.items():
    print(f"  * {clave}: {valor}")

# Entrenamiento

In [ ]:
mejores_parametros_rf = estudio_rf.best_params

modelo_rf_optimo = RandomForestClassifier(
    **mejores_parametros_rf,
    n_jobs=-1, 
    random_state=42
)

modelo_rf_optimo.fit(X_entrenamiento, y_entrenamiento)

# Umbral

In [ ]:
import numpy as np
from sklearn.metrics import fbeta_score, classification_report, confusion_matrix, roc_auc_score, f1_score

y_proba_prueba_rf = modelo_rf_optimo.predict_proba(X_prueba)[:, 1]

mejor_umbral_rf = 0.5
mejor_f2_rf = 0.0

print("Buscando el umbral óptimo\n")

for umbral in np.arange(0.01, 1.0, 0.01):
    prediccion_temporal = (y_proba_prueba_rf >= umbral).astype(int)
    
    f2_temporal = fbeta_score(y_prueba, prediccion_temporal, beta=2, zero_division=0)
    
    if f2_temporal > mejor_f2_rf:
        mejor_f2_rf = f2_temporal
        mejor_umbral_rf = umbral

print(f"Umbral ganador: {mejor_umbral_rf:.2f}")

# Desempeño

In [ ]:
y_pred_prueba_optimo_rf = (y_proba_prueba_rf >= mejor_umbral_rf).astype(int)

print("\nMatriz de Confusión Random Forest en Prueba:")
print(confusion_matrix(y_prueba, y_pred_prueba_optimo_rf))

print("\nDesempeño Random Forest en Prueba:")
print(classification_report(y_prueba, y_pred_prueba_optimo_rf, digits=5))

y_proba_oot_rf = modelo_rf_optimo.predict_proba(X_oot)[:, 1]
y_pred_oot_optimo_rf = (y_proba_oot_rf >= mejor_umbral_rf).astype(int)

print("\nMatriz de Confusión Random Forest OOT:")
print(confusion_matrix(y_oot, y_pred_oot_optimo_rf))

print("\nDesempeño Random Forest en OOT:")
print(classification_report(y_oot, y_pred_oot_optimo_rf, digits=5))